<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [Implementing a module →](02_Implementing_A_Module.ipynb)

# A module and its interfaces

A system architecture starts from its *modules*: the components you name, reuse, and instantiate. Before saying how
any module works inside, you say where it meets the world: its *interfaces*, the places where messages exist relative
to it, and each interface's *ports*, the sides or senses of it. A UART's `serial` interface has a transmit side and a
receive side; a memory's `bus` has a request side and a response side.

This tutorial models one module that way, reads it back, and finds it again by its path. It assumes the basics of
mbse-schemas (a store, transactions, builders that take a callback), which its
[first tutorial](https://github.com/pitaman71/mbse-schemas/blob/main/python3/tutorials/01_A_Contact_Card.ipynb)
teaches.

## The schemas live in your store

`Interactions.define(store)` builds the architecture schemas in a store you made: modules, interfaces, ports, and the
rest that later tutorials use. It runs outside any transaction, and running it again gives back what it built.

In [1]:
from mbse.Schemas.Framework import Builders
from mbse.Architecture.Systems.Framework import Interactions

store = Builders.Store()
I = Interactions.define(store)
print(I.Module.name, list(I.Module.adjacencies))
print(I.Interface.name, list(I.Interface.adjacencies))
print(I.Port.name, list(I.Port.properties))

Interactions.Module ['library', 'interfaces', 'implementations', 'instances']
Interactions.Interface ['module', 'ports', 'connections']
Interactions.Port ['direction', 'description']


## Every module lives in the library

A module is *addressable*: you find it again by name. So the store's one library, the singleton
`Interactions.Library`, owns every module under its name (the relation `Interactions.HasModule`). The module's name is
that entry's, not a property of the module, so it cannot disagree with the place the library keeps it.

Interfaces are owned by their module the same way (`Interactions.HasInterface(module, name, intf)`), and ports by
their interface (`Interactions.HasPort(intf, name, port)`). A port's `direction` is `"in"` or `"out"`, as its module
sees it.

In [2]:
library = store.singleton("Interactions.Library")

with store.transaction():
    store.Interactions.Library(library, lambda lib: lib.modules(lambda a: a.insert(lambda e: e.name("Uart").module(
        lambda m: m.description("An asynchronous serial port").interfaces(lambda a: a.insert(
            lambda e: e.name("serial").intf(lambda i: i.description("Bytes on the wire").ports(lambda p: (
                p
                .insert(lambda e: e.name("tx").port(lambda q: q.direction("out")))
                .insert(lambda e: e.name("rx").port(lambda q: q.direction("in")))
            )))
        ))
    ))))

uart = library.modules.at("Uart").module
serial = uart.interfaces.at("serial").intf
for entry in serial.ports:
    print(f"{entry.name}: {entry.port.direction}")

tx: out
rx: in


Reading goes through the same adjacencies: `at(name)` finds an entry by its key, and iterating gives every entry,
each with its name and the object it holds.

## Paths: where each part is

Since the library owns the module, the module owns its interfaces, and each interface its ports, every one of them has
a path from the library. `Interactions.path` writes it as dotted names, and the store finds an object by its keys,
one step at a time.

In [3]:
tx = serial.ports.at("tx").port
print([Interactions.path(x) for x in (uart, serial, tx)])

found = store.Instances.Interactions.Library.modules({"name": "Uart"}).interfaces({"name": "serial"}).ports({"name": "tx"})
print(found is tx)

['Uart', 'Uart.serial', 'Uart.serial.tx']
True


A module outside the library would have no path, so the store refuses one when its transaction ends, and writes
nothing of it.

In [4]:
try:
    with store.transaction():
        store.Interactions.Module(lambda m: m.description("Nobody's"))
except ValueError as error:
    print(error)
print([str(e.name) for e in library.modules])

a 'Interactions.Module' needs an owner: the key 'name' of 'Interactions.HasModule' has no naming pattern to give it a value
['Uart']


## Growing a module

A module changes as the design does. Its builder, given the module and a callback, adds to it: here a second
interface, for flow control, whose ports have the same names as `serial`'s, which is fine, since a port's name is
its interface's to give.

In [5]:
with store.transaction():
    store.Interactions.Module(uart, lambda m: m.interfaces(lambda a: a.insert(lambda e: e.name("flow").intf(
        lambda i: i.ports(lambda p: (
            p
            .insert(lambda e: e.name("tx").port(lambda q: q.direction("out").description("RTS")))
            .insert(lambda e: e.name("rx").port(lambda q: q.direction("in").description("CTS")))
        ))))))

for entry in uart.interfaces:
    print(entry.name, [f"{p.name}:{p.port.direction}" for p in entry.intf.ports])

serial ['tx:out', 'rx:in']
flow ['tx:out', 'rx:in']


Next, [Implementing a module](02_Implementing_A_Module.ipynb) says how a module works inside: instances of other
modules, and channels that carry messages between them.

<!-- nav -->
[← Python tutorial](README.md) · [Home](../../README.md) · [Implementing a module →](02_Implementing_A_Module.ipynb)